<a href="https://colab.research.google.com/github/anjaliii210/LLM-Post-Training-with-QLoRA-SFT-DPO/blob/main/0_env_model_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
import torch
import transformers
import datasets
import peft
import bitsandbytes
import trl

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

Torch: 2.14.1+cu130
Transformers: 5.22.2
Datasets: 5.2.0
PEFT: 0.21.2
BitsAndBytes: 0.50.2
TRL: 1.17.1


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Vocabulary size: 151665
EOS token: <|im_end|>
PAD token: <|endoftext|>
Chat template available: True


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Model loaded successfully")
print("Model device:", model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully
Model device: cuda:0


In [ ]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e

In [ ]:
print("Num attention heads:", model.config.num_attention_heads)
print("Num key/value heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)

# Calculate head_dim dynamically since it's not directly in the configuration
head_dim = getattr(model.config, "head_dim", model.config.hidden_size // model.config.num_attention_heads)
print("Head dimension:", head_dim)

Num attention heads: 14
Num key/value heads: 2
Hidden size: 896
Head dimension: 64


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Explain gradient descent with an example."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True
    )

print(tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain gradient descent with an example.
assistant
Gradient Descent is a method used to find the minimum of a function that takes as input variables and returns one or more outputs. In this case, we want to minimize a function called "cost" which represents the difference between two sets: a set of predictions made by our model and a target set of values.
The basic idea behind Gradient Descent is to update the parameters (weights) in our model based on how much they affect the cost. The process goes like this:

1. We start by initializing some random values for our parameters.

2. Then, we calculate the gradient of the cost function with respect to each parameter. This is done using a technique called backpropagation.

3. Next, we use the calculated gradients to update the


In [ ]:
text = "Explain gradient descent."

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

print("Input shape:", inputs["input_ids"].shape)
print("Logits shape:", outputs.logits.shape)

Input shape: torch.Size([1, 5])
Logits shape: torch.Size([1, 5, 151936])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/LLM_PostTraining/base_model"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
